# AI-Powered Sales Intelligence Agent
### XYZ Analytics Consulting — Indian Automotive Market


## 1. Installing dependencies

In [1]:
!pip install -q google-generativeai pypdf sentence-transformers tavily-python requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 7.5 MB/s eta 0:00:00


## 2. API keys
(Here Tavily API key was left blank and only Groq API Key was used)

In [2]:
!pip install -q groq pypdf sentence-transformers tavily-python requests

import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Enter Groq API key: ')
os.environ['TAVILY_API_KEY'] = getpass('Enter Tavily API key (or leave blank to skip live search): ')

from groq import Groq
client = Groq(api_key=os.environ['GROQ_API_KEY'])

def llm(prompt, temperature=0.4):
    resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return resp.choices[0].message.content

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.3 MB/s eta 0:00:00
Enter Groq API key: ··········
Enter Tavily API key (or leave blank to skip live search): ··········


## 3. Uploading & parsing the Product Handbook (RAG knowledge base)

In [3]:
from google.colab import files
print('Upload: XYZ_Analytics_Consulting___Product___Solutions_Handbook.pdf')
uploaded = files.upload()
handbook_path = list(uploaded.keys())[0]

Upload: XYZ_Analytics_Consulting___Product___Solutions_Handbook.pdf


Saving XYZ Analytics Consulting – Product & Solutions Handbook.pdf to XYZ Analytics Consulting – Product & Solutions Handbook.pdf


In [4]:
from pypdf import PdfReader

reader = PdfReader(handbook_path)
full_text = "\n".join(page.extract_text() or '' for page in reader.pages)

# Chunk by paragraph, ~250-400 words per chunk
def chunk_text(text, max_words=300):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words):
        chunks.append(' '.join(words[i:i+max_words]))
    return chunks

chunks = chunk_text(full_text)
print(f'Handbook parsed into {len(chunks)} chunks.')

Handbook parsed into 15 chunks.


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedder.encode(chunks, show_progress_bar=True)

def retrieve(query, k=3):
    q_emb = embedder.encode([query])[0]
    sims = chunk_embeddings @ q_emb / (np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(q_emb) + 1e-8)
    top_idx = np.argsort(sims)[::-1][:k]
    return [chunks[i] for i in top_idx]

## 4. Research tool: seed company list + optional live search
The seed list below covers major Indian OEMs, Tier-1 suppliers, and component manufacturers (publicly known — SIAM/ACMA member companies).

In [6]:
SEED_COMPANIES = [
    ("Maruti Suzuki India", "OEM - Passenger Vehicles"),
    ("Tata Motors", "OEM - PV/CV/EV"),
    ("Mahindra & Mahindra (Automotive)", "OEM - SUV/EV"),
    ("Bajaj Auto", "OEM - Two/Three-Wheelers"),
    ("TVS Motor Company", "OEM - Two-Wheelers"),
    ("Hero MotoCorp", "OEM - Two-Wheelers"),
    ("Ashok Leyland", "OEM - Commercial Vehicles"),
    ("VE Commercial Vehicles (Eicher)", "OEM - Commercial Vehicles"),
    ("Force Motors", "OEM - Commercial/Utility Vehicles"),
    ("Bosch Limited (India)", "Tier-1 - Powertrain/Electronics"),
    ("Samvardhana Motherson International", "Tier-1 - Wiring/Components"),
    ("Bharat Forge", "Tier-1 - Forgings/Components"),
    ("Sona BLW Precision Forgings (Sona Comstar)", "Tier-1 - EV Drivetrain"),
    ("UNO Minda", "Tier-1 - Switches/Electronics"),
    ("Endurance Technologies", "Tier-1 - Suspension/Braking"),
    ("Exide Industries", "Component - Batteries"),
    ("Amara Raja Energy & Mobility", "Component - Batteries"),
    ("MRF Limited", "Component - Tyres"),
    ("Apollo Tyres", "Component - Tyres"),
    ("Ola Electric Mobility", "OEM - Electric Two-Wheelers"),
]

def web_search(query, max_results=3):
    if not os.environ.get('TAVILY_API_KEY'):
        return []
    from tavily import TavilyClient
    client = TavilyClient(api_key=os.environ['TAVILY_API_KEY'])
    try:
        res = client.search(query=query, max_results=max_results)
        return [r['content'] for r in res.get('results', [])]
    except Exception as e:
        print('Search failed:', e)
        return []

## 5. Reasoning agent: profile + recommendation per company

In [7]:
PROFILE_PROMPT = """You are an automotive industry analyst. Based on general knowledge and the notes below, write a concise profile (4-6 sentences) of {company} ({segment}) covering: business scale, current operational challenges (warranty/quality, supply chain, or dealer/service related), and any recent strategic moves (EV transition, expansion, etc).

Recent notes (may be empty):
{notes}
"""

RECOMMEND_PROMPT = """You are a solutions consultant at XYZ Analytics Consulting. XYZ offers exactly THREE solutions:
1. Warranty Analytics — for companies with quality/defect/recall/claims issues
2. Supply-Chain Risk Prediction — for companies with multi-tier supplier networks, logistics, inventory, component sourcing issues
3. Dealer & Field Service Intelligence — for companies with dealer network, retail, customer service, aftersales issues

Given this company profile, pick the SINGLE best-fit solution based on what actually matters most for THIS company's business model (e.g. a tyre/battery/component maker's biggest exposure is usually supply-chain, not warranty; an OEM with large dealer networks may fit Dealer Intelligence). Do not default to Warranty Analytics unless the profile specifically shows quality/defect/recall issues.

Company Profile:
{profile}

Handbook Excerpts:
{handbook_context}

Respond in this exact format:
Recommended Solution: <name>
Justification: <text, must reference something specific from the profile that rules out the other two options>
Estimated Impact: <text>
"""

def build_company_record(company, segment):
    notes = web_search(f"{company} automotive India warranty OR supply chain OR dealer network news 2026")
    profile = llm(PROFILE_PROMPT.format(company=company, segment=segment, notes='\n'.join(notes) if notes else 'None available'))
    context_chunks = retrieve(profile, k=3)
    rec = llm(RECOMMEND_PROMPT.format(profile=profile, handbook_context='\n---\n'.join(context_chunks)))
    return {
        'company': company,
        'segment': segment,
        'profile': profile,
        'recommendation': rec
    }

In [8]:
results = []
for company, segment in SEED_COMPANIES:
    print(f'Processing: {company}...')
    try:
        record = build_company_record(company, segment)
        results.append(record)
    except Exception as e:
        print(f'  failed: {e}')

print(f'Done. {len(results)} companies processed.')

Processing: Maruti Suzuki India...
Processing: Tata Motors...
Processing: Mahindra & Mahindra (Automotive)...
Processing: Bajaj Auto...
Processing: TVS Motor Company...
Processing: Hero MotoCorp...
Processing: Ashok Leyland...
Processing: VE Commercial Vehicles (Eicher)...
Processing: Force Motors...
Processing: Bosch Limited (India)...
Processing: Samvardhana Motherson International...
Processing: Bharat Forge...
Processing: Sona BLW Precision Forgings (Sona Comstar)...
Processing: UNO Minda...
Processing: Endurance Technologies...
Processing: Exide Industries...
Processing: Amara Raja Energy & Mobility...
Processing: MRF Limited...
Processing: Apollo Tyres...
Processing: Ola Electric Mobility...
Done. 20 companies processed.


## 6. Generating final Market Research Report

In [13]:
REPORT_INTRO_PROMPT = """Write a 200-250 word executive market overview of the Indian automotive industry for a sales intelligence report, covering market size, growth drivers (PLI schemes, EV adoption), and the key operational challenges (warranty costs, supply chain complexity, dealer performance variability) that create demand for analytics consulting. Use this handbook context:
{context}
"""

market_context = '\n---\n'.join(retrieve('India automotive market size growth challenges', k=4))
market_overview = llm(REPORT_INTRO_PROMPT.format(context=market_context))

report_lines = [
    "# Sales Intelligence Report: Indian Automotive Analytics Opportunities",
    "## XYZ Analytics Consulting\n",
    "## 1. Market Research Report\n",
    market_overview,
    "\n## 2. Top Target Companies & Recommendations\n"
]

for r in results:
    report_lines.append(f"### {r['company']} ({r['segment']})")
    report_lines.append(f"**Profile:** {r['profile']}")
    report_lines.append(f"**Recommendation:** {r['recommendation']}\n")

final_report = '\n\n'.join(report_lines)

with open('Market_Research_Report.md', 'w') as f:
    f.write(final_report)

print(final_report)

# Sales Intelligence Report: Indian Automotive Analytics Opportunities

## XYZ Analytics Consulting


## 1. Market Research Report


Here is a 200-250 word executive market overview of the Indian automotive industry:

The Indian automotive industry is a significant contributor to the country's GDP, accounting for 7-8% of the national economy. With a market size of over $250 billion, the sector is poised for strong growth driven by government incentives such as PLI schemes and increasing adoption of electric vehicles (EVs). EV sales have been doubling year-on-year, with over 250 new models expected to enter the market in the coming years. However, original equipment manufacturers (OEMs) and suppliers face operational challenges, including high warranty costs, complex supply chains, and demand volatility. These challenges create a significant need for analytics consulting to optimize operations, reduce costs, and improve forecasting.

The Indian automotive industry is characterized by a 

## 7. Business recommendations summary

In [14]:
TOP_COMPANIES = [
    "Maruti Suzuki India", "Tata Motors", "Mahindra & Mahindra (Automotive)",
    "Bajaj Auto", "TVS Motor Company", "Hero MotoCorp", "Ashok Leyland",
    "Bosch Limited (India)", "Samvardhana Motherson International",
    "Bharat Forge", "Sona BLW Precision Forgings (Sona Comstar)",
    "UNO Minda", "Exide Industries", "Apollo Tyres", "Ola Electric Mobility",
]

top_results = [r for r in results if r['company'] in TOP_COMPANIES]
other_results = [r for r in results if r['company'] not in TOP_COMPANIES]

report_lines = [
    "# Sales Intelligence Report: Indian Automotive Analytics Opportunities",
    "## XYZ Analytics Consulting\n",
    "## 1. Market Research Report\n",
    market_overview,
    "\n## 2. Top 15 Target Companies & Recommendations\n"
]
for r in top_results:
    report_lines.append(f"### {r['company']} ({r['segment']})")
    report_lines.append(f"**Profile:** {r['profile']}")
    report_lines.append(f"**Recommendation:** {r['recommendation']}\n")

report_lines.append("\n## 3. Additional Companies Evaluated (Appendix)\n")
for r in other_results:
    report_lines.append(f"- **{r['company']}** ({r['segment']}) — see recommendation in full dataset")

final_report = '\n\n'.join(report_lines)
with open('Market_Research_Report.md', 'w') as f:
    f.write(final_report)

print(f'Top companies: {len(top_results)}, Appendix: {len(other_results)}')

Top companies: 15, Appendix: 5


In [15]:
SUMMARY_PROMPT = """You are preparing the closing section of a sales intelligence report for XYZ Analytics Consulting. Based on the individual company recommendations below, write a 250-300 word business recommendations section: which solution should XYZ prioritize selling first across this target list and why, which segment (OEM/Tier-1/component) looks most promising, and suggested outreach sequencing.

Recommendations:
{all_recs}
"""

all_recs_text = '\n'.join(f"{r['company']}: {r['recommendation']}" for r in results)
business_summary = llm(SUMMARY_PROMPT.format(all_recs=all_recs_text))

with open('Market_Research_Report.md', 'a') as f:
    f.write('\n\n## 3. Business Recommendations\n\n' + business_summary)

print(business_summary)

**Business Recommendations**

Based on the individual company recommendations, we suggest that XYZ Analytics Consulting prioritize selling **Warranty Analytics** as its first solution across the target list. This is because four out of the ten companies (Maruti Suzuki India, Tata Motors, Bajaj Auto, and Bosch Limited) have been recommended to implement Warranty Analytics, indicating a high demand for this solution in the market. Additionally, the estimated impact of implementing Warranty Analytics is significant, with potential reductions in warranty costs ranging from 5-10%, resulting in substantial savings and improved profitability.

In terms of segment, the **OEM** segment appears to be the most promising, with several companies (Maruti Suzuki India, Tata Motors, Bajaj Auto, and Bosch Limited) facing significant challenges related to warranty and quality issues. However, the **component** segment also presents opportunities, particularly for companies like Samvardhana Motherson Int

## 8. Downloading the report


In [16]:
from google.colab import files
files.download('Market_Research_Report.md')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>